In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_vendas = pd.read_csv("vendas_tech.csv", low_memory=False)
df_gerente = pd.read_excel("gerentes_lojas.xlsx")

In [64]:
df_analise = df_vendas.drop(columns=["Data_Base"])
df_analise['Loja'] = df_analise['Loja'].fillna("Online").str.strip().str.title()
df_gerente['Loja'] = df_gerente['Loja'].fillna("Online").str.strip().str.title()
df_analise['Data'] = pd.to_datetime(df_analise['Data'], format="%Y-%m-%d")
df_analise = df_analise.drop_duplicates(subset=["ID_Pedido"])


In [65]:
df_analise["Faturamento"] = df_analise["Preco_Unitario"] * df_analise["Qtd"]
df_analise["Forma_de_vendas"] = np.where(df_analise['Loja']=="Online","Online","Presencial" )

dic_regioes = {
    'São Paulo': 'Sudeste', 
    'Belo Horizonte': 'Sudeste',
    'Online': 'Online',
    'Rio De Janeiro': 'Sudeste',
    'Salvador': 'Nordeste',
    'Recife': 'Nordeste',
    'Curitiba': 'Sul',
    'Porto Alegre': 'Sul'
}

df_analise['Região'] = df_analise['Loja'].map(dic_regioes)



In [66]:
df_analise = df_analise.sort_values(by=["Data", "Faturamento"])
df_analise = df_analise.reset_index(drop=True)

id_pedido = 4

loja = df_analise.loc[df_analise["ID_Pedido"] == 4, "Loja"].values[0]
produto = df_analise.loc[df_analise["ID_Pedido"] == 4, "Produto"].values[0]
cliente = df_analise.loc[df_analise["ID_Pedido"] == 4, "Cliente"].values[0]

print(loja)
print(produto)
print(cliente)

id_pedido = 4

loja = df_analise.iloc[3, 2]
produto = df_analise.iloc[3, 3]
cliente = df_analise.iloc[3, 6]
#ranking de faturamento por loja
analise_lojas = df_analise[["Loja", "Faturamento"]].groupby("Loja").sum()
analise_lojas = analise_lojas.sort_values(by="Faturamento", ascending=False)
analise_lojas = analise_lojas.reset_index()
analise_lojas["Faturamento"] = analise_lojas["Faturamento"].map(
    lambda x: f"R${x:,.2f}"
)

df_vendas_online = df_analise[df_analise['Loja']=="Online"]
analise_produto_online = df_vendas_online[["Produto", "Faturamento"]].groupby("Produto").sum()

analise_produto_online = analise_produto_online.sort_values(by="Faturamento", ascending=False)


Rio De Janeiro
Mouse Gamer
Cliente_17343


In [67]:
df_meta = df_analise[(df_analise['Data'].dt.year==2023) & (df_analise['Data'].dt.month==1)]
df_meta = df_meta[['Faturamento', 'Loja']].groupby('Loja', as_index=False).sum()
df_meta = df_meta.merge(df_gerente, on='Loja', how="left")
df_meta['Meta_Atingida'] = np.where(df_meta['Faturamento'] >= df_meta['Meta_Mensal'], "Sim", "Não")


In [68]:
df_analise['Mes-Ano'] = df_analise['Data'].dt.to_period("M")
df_vendas_mes  = df_analise[['Mes-Ano', 'Faturamento']].groupby('Mes-Ano').sum()